In [ ]:
import numpy as np
import pandas as pd

# Movie Recommendation

## Data Loading and Data Preparation

In [19]:
def load_users():
    users = pd.read_csv('users.dat', sep='::', engine='python', 
                        names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'],
                        encoding='latin-1')
    # Convert Gender to binary: F=0, M=1
    users['Gender'] = (users['Gender'] == 'M').astype(int)
    # Drop Zip-code as per hint
    users = users.drop(columns=['Zip-code'])

    # Set UserID as index for easy lookup
    users = users.set_index('UserID')
    return users

def load_movies():
    movies = pd.read_csv('movies.dat', sep='::', engine='python',
                         names=['MovieID', 'Title', 'Genres'],
                         encoding='latin-1')
    # Make genre features numeric: first, define all possible genres
    all_genres = ['Action', 'Adventure', 'Animation', "Children's", 'Comedy',
                  'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
                  'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
                  'Thriller', 'War', 'Western']
    # Create binary columns for each genre
    for genre in all_genres:
        movies[f'Genre_{genre}'] = movies['Genres'].str.contains(genre, regex=False).astype(int)
    # Drop original Genres and Title columns
    movies = movies.drop(columns=['Title', 'Genres'])
   
    # Set MovieID as index for easy lookup
    movies = movies.set_index('MovieID')
    return movies

def load_ratings():
    ratings = pd.read_csv('ratings.dat', sep='::', engine='python',
                          names=['UserID', 'MovieID', 'Rating', 'Timestamp'],
                          encoding='latin-1')
    # Drop Timestamp as per hint
    ratings = ratings.drop(columns=['Timestamp'])
    return ratings

# Load the data
users = load_users()
movies = load_movies()
ratings = load_ratings()

print(f"Users loaded: {len(users)} users")
print(f"Movies loaded: {len(movies)} movies")
print(f"Ratings loaded: {len(ratings)} ratings")

Users loaded: 6040 users
Movies loaded: 3883 movies
Ratings loaded: 1000209 ratings


In [20]:
def filter_users_by_rating_count(ratings, min_ratings=100):
    # Remove users who have rated less than min_ratings movies.
    # Count ratings per user
    user_rating_counts = ratings.groupby('UserID').size()
    # Get users with at least min_ratings ratings
    valid_users = set(user_rating_counts[user_rating_counts >= min_ratings].index)
    # Filter ratings
    filtered_ratings = ratings[ratings['UserID'].isin(valid_users)]
    print(f"Users after filtering: {len(valid_users)}")
    return filtered_ratings, valid_users

# Filter users with less than 100 ratings
filtered_ratings, valid_users = filter_users_by_rating_count(ratings, min_ratings=100)

Users after filtering: 2945


In [21]:
def split_train_test(ratings, valid_users):
    # Split data into training and test sets based on user IDs.
    # Test data: all ratings from users with IDs 1 to 1000 (unless removed)
    # Training data: all remaining ratings
    test_user_ids = valid_users.intersection(set(range(1, 1001)))
    test_ratings = ratings[ratings['UserID'].isin(test_user_ids)]
    train_ratings = ratings[~ratings['UserID'].isin(test_user_ids)]
    return train_ratings, test_ratings, test_user_ids

# Split into train and test
train_ratings, test_ratings, test_user_ids = split_train_test(filtered_ratings, valid_users)

print(f"\nTraining ratings size: {len(train_ratings)}")
print(f"Test ratings size: {len(test_ratings)}")


Training ratings size: 719277
Test ratings size: 128025


In [22]:
def create_feature_vectors(ratings, users, movies):
    """
    Create feature vectors by combining user and movie information (optimized version).
    
    For each rating, create a feature vector containing:
    - User features: Gender (binary), Age (one-hot), Occupation (one-hot)
    - Movie features: Genre (binary for each of 18 genres)
    
    Returns:
    - X: Feature matrix (numpy array)
    - y: Target labels (ratings 1-5)
    - feature_names: List of feature names
    - data_full: DataFrame with all features and ratings
    """
    # Get unique ages and occupations for one-hot encoding
    age_values = sorted(users['Age'].unique())  # [1, 18, 25, 35, 45, 50, 56]
    occupation_values = sorted(users['Occupation'].unique())  # [0-20]
    
    # Create one-hot encoded user features
    users_encoded = users.copy()
    
    # One-hot encode Age
    for age in age_values:
        users_encoded[f'Age_{age}'] = (users['Age'] == age).astype(int)
    
    # One-hot encode Occupation
    for occ in occupation_values:
        users_encoded[f'Occupation_{occ}'] = (users['Occupation'] == occ).astype(int)
    
    # Drop original Age and Occupation columns
    users_encoded = users_encoded.drop(columns=['Age', 'Occupation'])
    
    # Reset index for merging
    users_encoded = users_encoded.reset_index()
    movies_reset = movies.reset_index()
    
    # Merge ratings with user features
    data = ratings.merge(users_encoded, on='UserID', how='inner')
    
    # Merge with movie features
    data = data.merge(movies_reset, on='MovieID', how='inner')
    
    # Get feature columns (exclude IDs and Rating)
    user_feature_cols = [col for col in users_encoded.columns if col != 'UserID']
    movie_feature_cols = [col for col in movies_reset.columns if col != 'MovieID']
    feature_cols = user_feature_cols + movie_feature_cols
    
    # Extract feature matrix and labels
    X = data[feature_cols].values
    y = data['Rating'].values
    
    return X, y, feature_cols, data

# Create feature vectors for training and test data
X_train, y_train, feature_names, train_data_full = create_feature_vectors(
    train_ratings, users, movies
)
X_test, y_test, _, test_data_full = create_feature_vectors(
    test_ratings, users, movies
)


In [23]:
# Display feature names
print("FEATURE DESCRIPTION")
print(f"{'='*60}")
print("\nUser Features:")
print("-" * 40)
user_features = [f for f in feature_names if not f.startswith('Genre_')]
for f in user_features:
    print(f"  - {f}")

print(f"\nMovie Features (Genre indicators):")
print("-" * 40)
movie_features = [f for f in feature_names if f.startswith('Genre_')]
for f in movie_features:
    print(f"  - {f}")
    
print(f"\nTotal: {len(user_features)} user features + {len(movie_features)} movie features = {len(feature_names)} features")

FEATURE DESCRIPTION

User Features:
----------------------------------------
  - Gender
  - Age_1
  - Age_18
  - Age_25
  - Age_35
  - Age_45
  - Age_50
  - Age_56
  - Occupation_0
  - Occupation_1
  - Occupation_2
  - Occupation_3
  - Occupation_4
  - Occupation_5
  - Occupation_6
  - Occupation_7
  - Occupation_8
  - Occupation_9
  - Occupation_10
  - Occupation_11
  - Occupation_12
  - Occupation_13
  - Occupation_14
  - Occupation_15
  - Occupation_16
  - Occupation_17
  - Occupation_18
  - Occupation_19
  - Occupation_20

Movie Features (Genre indicators):
----------------------------------------
  - Genre_Action
  - Genre_Adventure
  - Genre_Animation
  - Genre_Children's
  - Genre_Comedy
  - Genre_Crime
  - Genre_Documentary
  - Genre_Drama
  - Genre_Fantasy
  - Genre_Film-Noir
  - Genre_Horror
  - Genre_Musical
  - Genre_Mystery
  - Genre_Romance
  - Genre_Sci-Fi
  - Genre_Thriller
  - Genre_War
  - Genre_Western

Total: 29 user features + 18 movie features = 47 features


### Feature Vector Description

#### User Features (29 features)
- **Gender** (1 feature): Binary encoding (0 = Female, 1 = Male)
- **Age** (7 features): One-hot encoding of age groups (1, 18, 25, 35, 45, 50, 56)
- **Occupation** (21 features): One-hot encoding of occupation categories (0-20)

#### Movie Features (18 features)
- **Genre indicators** (18 features): Binary encoding for each genre (Action, Adventure, Animation, Children's, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western)

#### Feature Representation Rationale
1. **Binary encoding for Gender**: Gender has only two categories, so a single binary feature is sufficient and efficient.

2. **One-hot encoding for Age**: Age categories are ordinal but with irregular intervals (Under 18, 18-24, 25-34, etc.). One-hot encoding avoids imposing artificial numerical relationships between categories and allows the model to learn different patterns for each age group independently.

3. **One-hot encoding for Occupation**: Occupations are categorical with no inherent ordering. One-hot encoding ensures no artificial ordinality is imposed and allows the model to capture unique rating patterns for each profession.

4. **Binary genre indicators**: Movies can belong to multiple genres simultaneously. Using binary indicators for each genre allows representation of multi-label genre assignments (e.g., a movie can be both "Action" and "Comedy").


## Basic Movie Recommendation

In [24]:
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Use smaller subsamples for faster execution
np.random.seed(42)
subsample_size = 10000  # Reduced to 10k for CV tuning
subsample_indices = np.random.choice(len(X_train), size=subsample_size, replace=False)
X_train_sub = X_train[subsample_indices]
y_train_sub = y_train[subsample_indices]

# Also create a reduced training set for final model training
train_size = 50000  # Use 50k for final training instead of full 700k+
train_indices = np.random.choice(len(X_train), size=train_size, replace=False)
X_train_reduced = X_train[train_indices]
y_train_reduced = y_train[train_indices]

print(f"Using {subsample_size} samples for hyperparameter tuning (CV)")
print(f"Using {train_size} samples for final model training")
print(f"Full training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

Using 10000 samples for hyperparameter tuning (CV)
Using 50000 samples for final model training
Full training set: 719277 samples
Test set: 128025 samples


In [25]:
# First, compute the baseline: constant prediction accuracy
print("BASELINE: Constant Prediction Performance")
print("=" * 60)

# Calculate accuracy for predicting each constant class
for c in range(1, 6):
    train_acc = np.mean(y_train == c)
    test_acc = np.mean(y_test == c)
    print(f"Always predict {c}: Train accuracy = {train_acc:.4f}, Test accuracy = {test_acc:.4f}")

# Find the best constant prediction (mode)
best_constant = max(range(1, 6), key=lambda c: np.mean(y_test == c))
best_constant_acc = np.mean(y_test == best_constant)
print(f"\nBest constant prediction: Always predict {best_constant}")
print(f"Best constant accuracy on test set: {best_constant_acc:.4f} ({best_constant_acc*100:.2f}%)")

BASELINE: Constant Prediction Performance
Always predict 1: Train accuracy = 0.0586, Test accuracy = 0.0542
Always predict 2: Train accuracy = 0.1119, Test accuracy = 0.1125
Always predict 3: Train accuracy = 0.2683, Test accuracy = 0.2700
Always predict 4: Train accuracy = 0.3476, Test accuracy = 0.3536
Always predict 5: Train accuracy = 0.2137, Test accuracy = 0.2097

Best constant prediction: Always predict 4
Best constant accuracy on test set: 0.3536 (35.36%)


In [26]:
# Linear SVM with different regularization parameters C
print("LINEAR SVM HYPERPARAMETER TUNING")
print("=" * 60)

# C values to try (reduced to 3 values for speed)
C_values = [0.01, 0.1, 1.0]

svm_results = []
for C in C_values:
    print(f"\nTraining LinearSVC with C={C}...")
    svm = LinearSVC(C=C, max_iter=1000, random_state=42, dual='auto')
    
    # Cross-validation on subsample
    cv_scores = cross_val_score(svm, X_train_sub, y_train_sub, cv=3, scoring='accuracy')
    mean_cv = cv_scores.mean()
    std_cv = cv_scores.std()
    
    svm_results.append({
        'C': C,
        'cv_mean': mean_cv,
        'cv_std': std_cv
    })
    print(f"  CV Accuracy: {mean_cv:.4f} (+/- {std_cv:.4f})")

# Find best C
best_svm_result = max(svm_results, key=lambda x: x['cv_mean'])
print(f"\nBest SVM hyperparameter: C = {best_svm_result['C']}")
print(f"Best CV accuracy: {best_svm_result['cv_mean']:.4f}")

LINEAR SVM HYPERPARAMETER TUNING

Training LinearSVC with C=0.01...
  CV Accuracy: 0.3472 (+/- 0.0068)

Training LinearSVC with C=0.1...
  CV Accuracy: 0.3473 (+/- 0.0097)

Training LinearSVC with C=1.0...
  CV Accuracy: 0.3474 (+/- 0.0096)

Best SVM hyperparameter: C = 1.0
Best CV accuracy: 0.3474


In [27]:
# MLP Classifier with different hidden layer configurations
print("MLP CLASSIFIER HYPERPARAMETER TUNING")
print("=" * 60)

# Reduced configurations for faster execution
hidden_layer_configs = [
    (32,),              # 1 hidden layer with 32 neurons
    (32, 16),           # 2 hidden layers
    (64, 32),           # 2 hidden layers, larger
]

mlp_results = []
for hidden_layers in hidden_layer_configs:
    print(f"\nTraining MLP with hidden_layer_sizes={hidden_layers}...")
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        max_iter=100,  # Reduced iterations
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
    
    # Cross-validation on subsample
    cv_scores = cross_val_score(mlp, X_train_sub, y_train_sub, cv=3, scoring='accuracy')
    mean_cv = cv_scores.mean()
    std_cv = cv_scores.std()
    
    mlp_results.append({
        'hidden_layers': hidden_layers,
        'cv_mean': mean_cv,
        'cv_std': std_cv
    })
    print(f"  CV Accuracy: {mean_cv:.4f} (+/- {std_cv:.4f})")

# Find best configuration
best_mlp_result = max(mlp_results, key=lambda x: x['cv_mean'])
print(f"\nBest MLP hyperparameter: hidden_layer_sizes = {best_mlp_result['hidden_layers']}")
print(f"Best CV accuracy: {best_mlp_result['cv_mean']:.4f}")

MLP CLASSIFIER HYPERPARAMETER TUNING

Training MLP with hidden_layer_sizes=(32,)...
  CV Accuracy: 0.3391 (+/- 0.0060)

Training MLP with hidden_layer_sizes=(32, 16)...
  CV Accuracy: 0.3508 (+/- 0.0062)

Training MLP with hidden_layer_sizes=(64, 32)...
  CV Accuracy: 0.3465 (+/- 0.0054)

Best MLP hyperparameter: hidden_layer_sizes = (32, 16)
Best CV accuracy: 0.3508


In [28]:
# Train final models with best hyperparameters on reduced training data
print("FINAL MODEL TRAINING AND EVALUATION")
print("=" * 60)

# Train best SVM on reduced training data
print(f"\nTraining final LinearSVC with C={best_svm_result['C']}...")
best_svm = LinearSVC(C=best_svm_result['C'], max_iter=1000, random_state=42, dual='auto')
best_svm.fit(X_train_reduced, y_train_reduced)

# Evaluate on test set
y_pred_svm = best_svm.predict(X_test)
svm_test_acc = accuracy_score(y_test, y_pred_svm)
print(f"LinearSVC Test Accuracy: {svm_test_acc:.4f} ({svm_test_acc*100:.2f}%)")

FINAL MODEL TRAINING AND EVALUATION

Training final LinearSVC with C=1.0...
LinearSVC Test Accuracy: 0.3514 (35.14%)


In [29]:
# Train best MLP on reduced training data
print(f"\nTraining final MLP with hidden_layer_sizes={best_mlp_result['hidden_layers']}...")
best_mlp = MLPClassifier(
    hidden_layer_sizes=best_mlp_result['hidden_layers'],
    max_iter=100,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
best_mlp.fit(X_train_reduced, y_train_reduced)

# Evaluate on test set
y_pred_mlp = best_mlp.predict(X_test)
mlp_test_acc = accuracy_score(y_test, y_pred_mlp)
print(f"MLP Test Accuracy: {mlp_test_acc:.4f} ({mlp_test_acc*100:.2f}%)")


Training final MLP with hidden_layer_sizes=(32, 16)...
MLP Test Accuracy: 0.3498 (34.98%)


In [30]:
# Summary of all results
print("=" * 60)
print("SUMMARY OF RESULTS")
print("=" * 60)

print("\n1. BASELINE (Constant Prediction):")
print(f"   Best constant prediction: Always predict {best_constant}")
print(f"   Test accuracy: {best_constant_acc:.4f} ({best_constant_acc*100:.2f}%)")

print("\n2. LINEAR SVM RESULTS:")
print("   Hyperparameters tried (C values):", [r['C'] for r in svm_results])
print("   Cross-validation accuracies:")
for r in svm_results:
    print(f"     C={r['C']}: {r['cv_mean']:.4f} (+/- {r['cv_std']:.4f})")
print(f"   Best C: {best_svm_result['C']}")
print(f"   Test accuracy: {svm_test_acc:.4f} ({svm_test_acc*100:.2f}%)")

print("\n3. MLP CLASSIFIER RESULTS:")
print("   Hyperparameters tried (hidden_layer_sizes):")
for r in mlp_results:
    print(f"     {r['hidden_layers']}: {r['cv_mean']:.4f} (+/- {r['cv_std']:.4f})")
print(f"   Best hidden_layer_sizes: {best_mlp_result['hidden_layers']}")
print(f"   Test accuracy: {mlp_test_acc:.4f} ({mlp_test_acc*100:.2f}%)")

print("\n4. IMPROVEMENT OVER BASELINE:")
print(f"   LinearSVC improvement: {(svm_test_acc - best_constant_acc)*100:.2f} percentage points")
print(f"   MLP improvement: {(mlp_test_acc - best_constant_acc)*100:.2f} percentage points")

SUMMARY OF RESULTS

1. BASELINE (Constant Prediction):
   Best constant prediction: Always predict 4
   Test accuracy: 0.3536 (35.36%)

2. LINEAR SVM RESULTS:
   Hyperparameters tried (C values): [0.01, 0.1, 1.0]
   Cross-validation accuracies:
     C=0.01: 0.3472 (+/- 0.0068)
     C=0.1: 0.3473 (+/- 0.0097)
     C=1.0: 0.3474 (+/- 0.0096)
   Best C: 1.0
   Test accuracy: 0.3514 (35.14%)

3. MLP CLASSIFIER RESULTS:
   Hyperparameters tried (hidden_layer_sizes):
     (32,): 0.3391 (+/- 0.0060)
     (32, 16): 0.3508 (+/- 0.0062)
     (64, 32): 0.3465 (+/- 0.0054)
   Best hidden_layer_sizes: (32, 16)
   Test accuracy: 0.3498 (34.98%)

4. IMPROVEMENT OVER BASELINE:
   LinearSVC improvement: -0.22 percentage points
   MLP improvement: -0.38 percentage points


### Results Summary

Interestingly, we did not succeed in finding a better MLP classifier compared to the (rather stupid)
approach of just always taking a 4 rating.

This suggests, that the features we have, like age and education, does not really contribute in
predicting a movie rating.


## Classifier Evaluation

In [31]:
def compute_confusion_matrix(y_true, y_pred, classes):
    n_classes = len(classes)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    
    # Create mapping from class label to index
    class_to_idx = {c: i for i, c in enumerate(classes)}
    
    # Count occurrences
    for true_label, pred_label in zip(y_true, y_pred):
        true_idx = class_to_idx[true_label]
        pred_idx = class_to_idx[pred_label]
        cm[true_idx, pred_idx] += 1
    
    return cm, classes


def print_confusion_matrix(cm, classes):
    # Header
    header = "True\\Pred |"
    for c in classes:
        header += f"   {c}   |"
    print(header)
    print("-" * len(header))
    # Data
    for i, true_class in enumerate(classes):
        row = f"    {true_class}     |"
        for j in range(len(classes)):
            row += f" {cm[i,j]:5d} |"
        print(row)
    
    print("-" * len(header))


def compute_accuracy_from_confusion_matrix(cm):
    """
    Compute classification accuracy from confusion matrix.
    
    Accuracy = (sum of diagonal elements) / (sum of all elements)
             = (number of correct predictions) / (total predictions)
    """
    correct = np.trace(cm)  # Sum of diagonal (correct predictions)
    total = np.sum(cm)      # Total predictions
    accuracy = correct / total
    return accuracy, correct, total

In [32]:
# Compute and display confusion matrix for LinearSVC
print("Confusion matrix for SVM\n")
classes = [1, 2, 3, 4, 5]

cm_svm, _ = compute_confusion_matrix(y_test, y_pred_svm, classes)
print_confusion_matrix(cm_svm, classes)

# Compute accuracy from confusion matrix
acc_svm, correct_svm, total_svm = compute_accuracy_from_confusion_matrix(cm_svm)
print(f"\nAccuracy computation from confusion matrix:")
print(f"  Correct predictions (diagonal sum): {correct_svm}")
print(f"  Total predictions: {total_svm}")
print(f"  Accuracy = {correct_svm} / {total_svm} = {acc_svm:.4f} ({acc_svm*100:.2f}%)")

Confusion matrix for SVM

True\Pred |   1   |   2   |   3   |   4   |   5   |
---------------------------------------------------
    1     |     0 |     0 |  1176 |  5719 |    41 |
    2     |     0 |     0 |  1713 | 12575 |   117 |
    3     |     0 |     0 |  3597 | 30562 |   411 |
    4     |     0 |     0 |  3859 | 40610 |   801 |
    5     |     0 |     0 |  2000 | 24063 |   781 |
---------------------------------------------------

Accuracy computation from confusion matrix:
  Correct predictions (diagonal sum): 44988
  Total predictions: 128025
  Accuracy = 44988 / 128025 = 0.3514 (35.14%)


In [33]:
# Compute and display confusion matrix for MLP
print("Confusion matrix for MLP\n")

cm_mlp, _ = compute_confusion_matrix(y_test, y_pred_mlp, classes)
print_confusion_matrix(cm_mlp, classes)

# Compute accuracy from confusion matrix
acc_mlp, correct_mlp, total_mlp = compute_accuracy_from_confusion_matrix(cm_mlp)
print(f"\nAccuracy computation from confusion matrix:")
print(f"  Correct predictions (diagonal sum): {correct_mlp}")
print(f"  Total predictions: {total_mlp}")
print(f"  Accuracy = {correct_mlp} / {total_mlp} = {acc_mlp:.4f} ({acc_mlp*100:.2f}%)")

Confusion matrix for MLP

True\Pred |   1   |   2   |   3   |   4   |   5   |
---------------------------------------------------
    1     |     0 |     0 |   908 |  5924 |   104 |
    2     |     0 |     0 |  1365 | 12794 |   246 |
    3     |     0 |     0 |  2948 | 30875 |   747 |
    4     |     0 |     0 |  3383 | 40517 |  1370 |
    5     |     0 |     0 |  1841 | 23684 |  1319 |
---------------------------------------------------

Accuracy computation from confusion matrix:
  Correct predictions (diagonal sum): 44784
  Total predictions: 128025
  Accuracy = 44784 / 128025 = 0.3498 (34.98%)


We do not see any specific structure in the confusion matrix. It just seems strange that there is no prediction for 1 and 2 ratings.